## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Libraries imported successfully!')

In [ ]:
# Load the restaurant reviews dataset
df = pd.read_csv('../archive (1)/top 240 restaurants recommanded in los angeles 2.csv')

print(f'Dataset Shape: {df.shape}')
print(f'\nColumn Names:\n{df.columns.tolist()}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Basic data exploration
print('Data Info:')
print(df.info())
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nBasic Statistics:')
df.describe()

## 2. Text Preprocessing

In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

print('NLTK resources downloaded!')

In [ ]:
def clean_text(text):
    """Clean and preprocess review text"""
    if pd.isna(text):
        return ''
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Normalize unicode characters (e.g., Â)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning
df['cleaned_text'] = df['Comment'].apply(clean_text)

print('Text cleaned! Sample comparison:')
for i in range(3):
    print(f"\nOriginal: {df['Comment'].iloc[i][:100]}...")
    print(f"Cleaned: {df['cleaned_text'].iloc[i][:100]}...")

In [ ]:
def tokenize_and_lemmatize(text):
    """Tokenize and lemmatize text"""
    if not text:
        return []
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return tokens

# Apply tokenization and lemmatization
df['tokens'] = df['cleaned_text'].apply(tokenize_and_lemmatize)

# Create a column with rejoined tokens (for analysis)
df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))

print('Tokenization and lemmatization complete!')
print(f"\nSample processed text (first review):")
print(df['processed_text'].iloc[0])

## 3. Exploratory Text Analysis

In [ ]:
# Calculate review statistics
df['review_length'] = df['cleaned_text'].apply(lambda x: len(x.split()))
df['token_count'] = df['tokens'].apply(len)

print('Review Length Statistics:')
print(df['review_length'].describe())

# Visualize review length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['review_length'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Review Length (words)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Review Lengths')

axes[1].hist(df['token_count'], bins=50, color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Token Counts (after preprocessing)')

plt.tight_layout()
plt.show()

print(f'\nAverage review length: {df["review_length"].mean():.2f} words')
print(f'Average token count: {df["token_count"].mean():.2f} tokens')

In [ ]:
from collections import Counter

# Get overall word frequencies
all_tokens = [token for tokens_list in df['tokens'] for token in tokens_list]
word_freq = Counter(all_tokens)
top_words = word_freq.most_common(30)

# Visualize top words
top_words_df = pd.DataFrame(top_words, columns=['word', 'frequency'])

plt.figure(figsize=(14, 6))
sns.barplot(data=top_words_df, x='frequency', y='word', palette='viridis')
plt.title('Top 30 Most Frequent Words in Reviews', fontsize=16, fontweight='bold')
plt.xlabel('Frequency')
plt.show()

print('Top 30 Words:')
print(top_words_df)

In [ ]:
# Generate word cloud
text_for_wordcloud = ' '.join(df['processed_text'].values)

wordcloud = WordCloud(width=1200, height=600, 
                      background_color='white',
                      colormap='viridis',
                      max_words=100).generate(text_for_wordcloud)

plt.figure(figsize=(16, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Restaurant Reviews', fontsize=16, fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

## 4. Sentiment Analysis

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    """Get sentiment polarity and subjectivity using TextBlob"""
    if not text or len(text) < 3:
        return 0, 0
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

# Apply sentiment analysis
df[['sentiment_polarity', 'sentiment_subjectivity']] = df['cleaned_text'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

# Classify sentiment
def classify_sentiment(polarity):
    if polarity > 0.1:
        return 'Positive'
    elif polarity < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_label'] = df['sentiment_polarity'].apply(classify_sentiment)

print('Sentiment Analysis Complete!')
print(f"\nSentiment Distribution:")
print(df['sentiment_label'].value_counts())
print(f"\nSentiment Statistics:")
print(df['sentiment_polarity'].describe())

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment label counts
sentiment_counts = df['sentiment_label'].value_counts()
axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=['green', 'red', 'gray'])
axes[0].set_ylabel('Count')
axes[0].set_title('Sentiment Label Distribution')
axes[0].set_ylim(0, max(sentiment_counts.values) * 1.1)

# Sentiment polarity distribution
axes[1].hist(df['sentiment_polarity'], bins=50, color='skyblue', edgecolor='black')
axes[1].set_xlabel('Polarity Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Sentiment Polarity Distribution')
axes[1].axvline(x=0, color='red', linestyle='--', label='Neutral')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sentiment by restaurant (top 15)
restaurant_sentiment = df.groupby('RestaurantName').agg({
    'sentiment_polarity': 'mean',
    'RestaurantName': 'count'
}).rename(columns={'RestaurantName': 'review_count'})

restaurant_sentiment = restaurant_sentiment.sort_values('sentiment_polarity', ascending=False).head(15)

plt.figure(figsize=(14, 8))
sns.barplot(data=restaurant_sentiment.reset_index(), x='sentiment_polarity', y='RestaurantName', palette='coolwarm')
plt.xlabel('Average Sentiment Polarity')
plt.title('Top 15 Restaurants by Average Sentiment', fontsize=14, fontweight='bold')
plt.show()

print('Top 15 Restaurants by Average Sentiment:')
print(restaurant_sentiment)

In [ ]:
# Sentiment vs Star Rating correlation
correlation = df['sentiment_polarity'].corr(df['StarRating'])

plt.figure(figsize=(10, 6))
plt.scatter(df['sentiment_polarity'], df['StarRating'], alpha=0.3, s=30)
plt.xlabel('Sentiment Polarity')
plt.ylabel('Star Rating')
plt.title(f'Sentiment Polarity vs Star Rating (Correlation: {correlation:.3f})', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Correlation between Sentiment and Star Rating: {correlation:.4f}')

## 5. Keyword Extraction with TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=100, min_df=5, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(df['processed_text'])

# Get feature names
feature_names = tfidf.get_feature_names_out()

print('TF-IDF vectorization complete!')
print(f'Matrix shape: {tfidf_matrix.shape}')
print(f'Top features: {feature_names[:20].tolist()}')

In [ ]:
# Get top TF-IDF terms overall
tfidf_means = tfidf_matrix.mean(axis=0).A1
top_tfidf_idx = tfidf_means.argsort()[-20:][::-1]
top_tfidf_terms = [(feature_names[i], tfidf_means[i]) for i in top_tfidf_idx]

top_tfidf_df = pd.DataFrame(top_tfidf_terms, columns=['term', 'tfidf_score'])

plt.figure(figsize=(12, 6))
sns.barplot(data=top_tfidf_df, x='tfidf_score', y='term', palette='rocket')
plt.xlabel('TF-IDF Score')
plt.title('Top 20 TF-IDF Terms', fontsize=14, fontweight='bold')
plt.show()

print('Top 20 TF-IDF Terms:')
print(top_tfidf_df)

## 6. Compare TF-IDF: Positive vs Negative Reviews

In [ ]:
# Separate positive and negative reviews
positive_reviews = df[df['sentiment_label'] == 'Positive']['processed_text']
negative_reviews = df[df['sentiment_label'] == 'Negative']['processed_text']

# TF-IDF for positive reviews
tfidf_pos = TfidfVectorizer(max_features=50, min_df=2, max_df=0.8)
tfidf_matrix_pos = tfidf_pos.fit_transform(positive_reviews)
feature_names_pos = tfidf_pos.get_feature_names_out()
tfidf_means_pos = tfidf_matrix_pos.mean(axis=0).A1

# TF-IDF for negative reviews
tfidf_neg = TfidfVectorizer(max_features=50, min_df=2, max_df=0.8)
tfidf_matrix_neg = tfidf_neg.fit_transform(negative_reviews)
feature_names_neg = tfidf_neg.get_feature_names_out()
tfidf_means_neg = tfidf_matrix_neg.mean(axis=0).A1

print(f'Positive reviews: {len(positive_reviews)}')
print(f'Negative reviews: {len(negative_reviews)}')

## 7. Word Embeddings

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec model
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec model trained!')
print(f'Vocabulary size: {len(w2v_model.wv)}')
print(f'Vector size: {w2v_model.vector_size}')

In [ ]:
def get_embedding_vector(tokens, model, vector_size=100):
    """Get average embedding for a list of tokens"""
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)

# Create embedding vectors for each review
df['embedding'] = df['tokens'].apply(lambda x: get_embedding_vector(x, w2v_model))

# Convert embeddings to DataFrame for clustering
embeddings_array = np.array(df['embedding'].tolist())

print('Embedding vectors created!')
print(f'Embedding shape: {embeddings_array.shape}')
print(f'Sample embedding vector (first 10 dims): {embeddings_array[0][:10]}')

## 8. Feature Engineering from Embeddings

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Standardize embeddings
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_array)

# Determine optimal number of clusters using elbow method
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Apply k-means clustering with k=5
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['review_cluster'] = kmeans.fit_predict(embeddings_scaled)

print(f'K-Means clustering complete with k={optimal_k}')
print(f'\nCluster distribution:')
print(df['review_cluster'].value_counts().sort_index())

In [ ]:
# Analyze clusters - show key characteristics
cluster_analysis = df.groupby('review_cluster').agg({
    'sentiment_polarity': 'mean',
    'review_length': 'mean',
    'StarRating': 'mean',
    'RestaurantName': 'count'
}).rename(columns={'RestaurantName': 'review_count'})

print('Cluster Characteristics:')
print(cluster_analysis)

# Show sample reviews from each cluster
print('\n\nSample Reviews by Cluster:\n')
for cluster_id in range(optimal_k):
    sample_review = df[df['review_cluster'] == cluster_id]['cleaned_text'].iloc[0]
    print(f"Cluster {cluster_id}: {sample_review[:150]}...")
    print()

## 9. Clustering and Segmentation

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Create a document-term matrix for LDA
vectorizer = CountVectorizer(max_features=100, min_df=5, max_df=0.8, stop_words='english')
dtm = vectorizer.fit_transform(df['processed_text'])

print('Document-Term Matrix created')
print(f'Shape: {dtm.shape}')

# Train LDA model
n_topics = 6
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=10)
lda_output = lda_model.fit_transform(dtm)

df['lda_topics'] = lda_output.argmax(axis=1)

print(f'\nLDA model trained with {n_topics} topics')
print(f'\nTopic distribution:')
print(df['lda_topics'].value_counts().sort_index())

In [ ]:
# Display topics and their top words
terms = vectorizer.get_feature_names_out()

print('Top words for each topic:\n')
for topic_idx, topic in enumerate(lda_model.components_):
    top_words_idx = topic.argsort()[-10:][::-1]
    top_words = [terms[i] for i in top_words_idx]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

# Visualize topic distribution
fig, ax = plt.subplots(figsize=(10, 6))
topic_counts = df['lda_topics'].value_counts().sort_index()
ax.bar(topic_counts.index, topic_counts.values, color='skyblue', edgecolor='black')
ax.set_xlabel('Topic ID')
ax.set_ylabel('Number of Reviews')
ax.set_title('Distribution of Reviews Across LDA Topics', fontsize=14, fontweight='bold')
ax.set_xticks(range(n_topics))
plt.show()

## 10. Topic Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix

# Create feature matrix combining sentiment, embeddings, and topic proportions
X_features = np.hstack([
    df['sentiment_polarity'].values.reshape(-1, 1),
    embeddings_array,
    lda_output  # Topic proportions
])

# Target variable: Star Rating
y_rating = df['StarRating'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_features, y_rating, test_size=0.2, random_state=42)

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('Random Forest Regression Results:')
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'R² Score: {r2:.4f}')

# Visualize predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Star Rating')
plt.ylabel('Predicted Star Rating')
plt.title('Predicted vs Actual Star Ratings', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Feature importance
feature_names_combined = ['sentiment'] + [f'embedding_{i}' for i in range(100)] + [f'topic_{i}' for i in range(n_topics)]
importances = rf_model.feature_importances_
top_features_idx = importances.argsort()[-15:][::-1]

plt.figure(figsize=(12, 6))
plt.barh([feature_names_combined[i] for i in top_features_idx], importances[top_features_idx])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features for Rating Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Predictive Modeling

In [ ]:
# === BUSINESS INSIGHTS & RECOMMENDATIONS ===\n\nprint('='*70)\nprint('BUSINESS INSIGHTS & RECOMMENDATIONS FOR LA RESTAURANT MANAGERS')\nprint('='*70)\n\n# 1. Key drivers of high ratings\nprint('\\n1. KEY DRIVERS OF HIGH RATINGS:')\nprint(f'   - Sentiment analysis shows {(df[df[\"StarRating\"] >= 4.5][\"sentiment_label\"] == \"Positive\").sum() / len(df[df[\"StarRating\"] >= 4.5]) * 100:.1f}% of 5-star reviews are positive sentiment')\nprint(f'   - Average polarity for 5-star reviews: {df[df[\"StarRating\"] >= 4.5][\"sentiment_polarity\"].mean():.3f}')\nprint(f'   - Average polarity for 1-star reviews: {df[df[\"StarRating\"] == 1.0][\"sentiment_polarity\"].mean():.3f}')\n\n# 2. Common themes in positive vs negative\nprint('\\n2. TOP THEMES IN POSITIVE REVIEWS:')\npositive_df = df[df['sentiment_label'] == 'Positive']\npositive_text = ' '.join(positive_df['processed_text'].values)\npos_word_freq = Counter([token for tokens_list in positive_df['tokens'] for token in tokens_list])\nfor word, count in pos_word_freq.most_common(10):\n    print(f'   - {word}: {count} mentions')\n\nprint('\\n3. TOP THEMES IN NEGATIVE REVIEWS:')\nnegative_df = df[df['sentiment_label'] == 'Negative']\nnegative_text = ' '.join(negative_df['processed_text'].values)\nneg_word_freq = Counter([token for tokens_list in negative_df['tokens'] for token in tokens_list])\nfor word, count in neg_word_freq.most_common(10):\n    print(f'   - {word}: {count} mentions')\n\n# 3. Restaurant-level recommendations\nprint('\\n4. TOP PERFORMING RESTAURANTS (by average sentiment):')\ntop_restaurants = df.groupby('RestaurantName').agg({\n    'sentiment_polarity': 'mean',\n    'StarRating': 'mean',\n    'RestaurantName': 'count'\n}).rename(columns={'RestaurantName': 'review_count'}).sort_values('sentiment_polarity', ascending=False).head(5)\nfor idx, (rest, row) in enumerate(top_restaurants.iterrows(), 1):\n    print(f'   {idx}. {rest}: Sentiment={row[\"sentiment_polarity\"]:.3f}, Avg Rating={row[\"StarRating\"]:.2f}, Reviews={int(row[\"review_count\"])}')\n\n# 4. Areas for improvement\nprint('\\n5. AREAS FOR IMPROVEMENT (Restaurants with lowest sentiment):')\nlow_performers = df.groupby('RestaurantName').agg({\n    'sentiment_polarity': 'mean',\n    'StarRating': 'mean',\n    'RestaurantName': 'count'\n}).rename(columns={'RestaurantName': 'review_count'}).sort_values('sentiment_polarity', ascending=True).head(5)\nfor idx, (rest, row) in enumerate(low_performers.iterrows(), 1):\n    print(f'   {idx}. {rest}: Sentiment={row[\"sentiment_polarity\"]:.3f}, Avg Rating={row[\"StarRating\"]:.2f}')\n\nprint('\\n' + '='*70)\nprint('ACTIONABLE RECOMMENDATIONS')\nprint('='*70)\nprint('''\n1. STAFFING & SERVICE:\n   - Focus on staff training to improve service quality (frequent theme in negative reviews)\n   - Implement customer service protocols to address common complaints\n   - Monitor wait times and staffing during peak hours\n\n2. MENU & FOOD QUALITY:\n   - Analyze positive reviews to identify best-selling dishes\n   - Highlight frequently praised menu items in marketing\n   - Address consistency issues mentioned in negative reviews\n\n3. ATMOSPHERE & AMBIANCE:\n   - Create welcoming environment (major driver of positive sentiment)\n   - Address noise/cleanliness concerns from negative reviews\n   - Invest in ambiance improvements for casual dining establishments\n\n4. PRICING STRATEGY:\n   - Review pricing feedback in reviews\n   - Ensure value perception matches price point\n   - Consider promotional strategies for low-sentiment segments\n\n5. MARKETING INSIGHTS:\n   - Leverage identified key themes for marketing campaigns\n   - Highlight top drivers of satisfaction in promotional materials\n   - Use sentiment analysis to monitor brand perception\n''')\n\nprint('\\nModel Performance Summary:')\nprint(f'- Sentiment-Rating Correlation: {correlation:.4f}')\nprint(f'- Predictive Model R² Score: {r2:.4f}')\nprint(f'- Average Review Sentiment: {df[\"sentiment_polarity\"].mean():.3f}')\nprint(f'- Overall Dataset Size: {len(df)} reviews from {df[\"RestaurantName\"].nunique()} restaurants')"


## 12. Business Insights & Recommendations